# RF-DETR-M (COCO-чекпойнт) на 5 семейных фолдах (bot-5)
**Честный замер OOF.** По умолчанию `--init coco`: весь детектор из `rf-detr-medium.pth` (Roboflow, Apache-2.0, md5 `7223f764a87b863f02eb8d52bf0ce2ee`; веса COCO разрешены организаторами), голова классов (5+1) — заново. Код: `scripts/rfdetr_train.py` (свой цикл; из rfdetr 1.11.2 — сборка модели, матчер, критерий, `load_pretrain_weights`). Подробности — `kaggle/README_KAGGLE_RFDETR.md`.

Inputs: `expds-koz3-bot5` (репо, ветка `bot5-rfdetr-coco`), `koz3-spor-det05` (`f0..f4`, `grid_f0..grid_f4`). GPU T4×2, Internet ON (pip + чекпойнт с storage.googleapis.com; без интернета — приложить Dataset с `rf-detr-medium.pth` и задать `COCO_W`).

**Какой чекпойнт (29.10, разрешены веса Objects365/IN-21k).** Чистого O365-чекпойнта размера M у Roboflow нет (реестр `rfdetr/assets/model_weights.py` 1.11.2). По статье RF-DETR (arXiv 2511.09554, табл. 5 «Additional O365 Pre-Training», прил. A) детекторы RF-DETR предобучены на Objects365 и затем на COCO, т.е. `rf-detr-medium.pth` = O365→COCO — **вариант A, основной**. Вариант B (ближайший размер с чистым O365): `rf-detr-base-o365.pth` (Base, DINOv2-S, патч 14), `https://storage.googleapis.com/rfdetr/top-secret-1234/lwdetr_dinov2_small_o365_checkpoint.pth`, md5 `d93f4921ccbb0f0a2e4364bed290892b`, Apache-2.0 — запускать только после A, `MODEL=rfdetr-b SIZE=672`. Запасной: `INIT=dinov2` (только бэкбон).

In [ ]:
REPO='/kaggle/input/expds-koz3-bot5'   # поправить на фактический путь датасета
!pip install -q --no-deps "rfdetr==1.11.2" 2>&1 | tail -1
!pip install -q -r $REPO/kaggle/requirements_rfdetr.txt 2>&1 | tail -1  # transformers==5.21.0 и др. (точно)
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count())
assert transformers.__version__ == '5.21.0', transformers.__version__

In [ ]:
import os, hashlib, urllib.request
REPO='/kaggle/input/expds-koz3-bot5'   # поправить на фактический путь датасета
DET='/kaggle/input/koz3-spor-det05'
OUT='/kaggle/working/rfdetr'
os.makedirs(OUT, exist_ok=True)
COCO_W='/kaggle/working/rf-detr-medium.pth'   # или путь в приложенном Dataset
if not os.path.exists(COCO_W):
    urllib.request.urlretrieve('https://storage.googleapis.com/rfdetr/medium_coco/checkpoint_best_regular.pth', COCO_W)
md5=hashlib.md5(open(COCO_W,'rb').read()).hexdigest(); print(md5)
assert md5=='7223f764a87b863f02eb8d52bf0ce2ee', md5   # = rfdetr/assets/model_weights.py RF_DETR_MEDIUM
os.environ['COCO_W']=COCO_W; os.environ['INIT']='coco'
!ls $DET

## Шаг 1 — smoke (10 итераций, обязателен первым). Только честный замер скорости/памяти.
Проверить в логе `coco: {... 'md5': '7223f764…', 'tensors': 510, 'changed': 508, 'head_out': 6}` (так на боксе, `bot-5/tmp/rf_coco_bs1/timing.json`). Записать `s_per_iter`, `cuda_peak_MB`. Затем замер памяти bs 4 (`--iters 20 --bs 4 --accum 1`).

In [ ]:
!python $REPO/scripts/rfdetr_train.py --data $DET/f0 --out $OUT/smoke --model rfdetr-m --size 640 --fp16 --init coco --coco-weights $COCO_W --smoke
!cat $OUT/smoke/timing.json
!python $REPO/scripts/rfdetr_train.py --data $DET/f0 --out $OUT/mem_bs4 --model rfdetr-m --size 640 --fp16 --init coco --coco-weights $COCO_W --iters 20 --bs 4 --accum 1 --warmup-iters 5 --log-every 5 --pred-split none 2>&1 | tail -3

Бюджет полосы GPU0 (3 фолда): `s_per_iter(bs4) × accum 4 × ITERS × 3` + сетка. Если > ~11 ч — `ITERS=6000` или фолды по сессиям. OOM на bs 4 → `--bs 2 --accum 8`.

In [ ]:
ITERS=8000
!INIT=coco COCO_W=$COCO_W bash $REPO/kaggle/rfdetr_folds.sh $DET $OUT rfdetr-m $ITERS

In [ ]:
!for k in 0 1 2 3 4; do cat $OUT/f$k/timing.json | python -c "import json,sys;d=json.load(sys.stdin);print($k,{x:d.get(x) for x in ['s_per_iter','train_s','cuda_peak_MB','loss_first','loss_last']})"; done
!cd /kaggle/working && tar czf rfdetr_preds.tgz rfdetr/f*/grid/preds_valid.json rfdetr/f*/timing.json rfdetr/f*/grid/timing.json rfdetr/*.log && ls -la rfdetr_preds.tgz

Скачать `rfdetr_preds.tgz`. Дальше — на машине с данными: `README_KAGGLE_RFDETR.md`, шаг 4 (coco_preds_to_stack `--group sp_rfdetr` → scoreboard --honest 10 → П1–П6).

## Вариант B (необязательно, после A): чистый Objects365, RF-DETR-Base
Отдельная сессия, `OUT` другой. Сторона кратна 56 (672).

In [ ]:
O365_W='/kaggle/working/rf-detr-base-o365.pth'
if not os.path.exists(O365_W):
    urllib.request.urlretrieve('https://storage.googleapis.com/rfdetr/top-secret-1234/lwdetr_dinov2_small_o365_checkpoint.pth', O365_W)
md5=hashlib.md5(open(O365_W,'rb').read()).hexdigest(); print(md5)
assert md5=='d93f4921ccbb0f0a2e4364bed290892b', md5   # = model_weights.py RF_DETR_BASE_O365 и x-goog-hash GCS
OUTB='/kaggle/working/rfdetr_b365'
!python $REPO/scripts/rfdetr_train.py --data $DET/f0 --out $OUTB/smoke --model rfdetr-b --size 672 --fp16 --init coco --coco-weights $O365_W --smoke
# в логе: coco: {... 'tensors': 488, 'changed': 486, 'head_out': 6}
!INIT=coco COCO_W=$O365_W SIZE=672 bash $REPO/kaggle/rfdetr_folds.sh $DET $OUTB rfdetr-b $ITERS
!cd /kaggle/working && tar czf rfdetr_b365_preds.tgz rfdetr_b365/f*/grid/preds_valid.json rfdetr_b365/f*/timing.json rfdetr_b365/*.log
